# Blood Cell Detection using CNN

**Dataset:** Blood Cell Detection Dataset by draaslan  
**Task:** Binary image classification of cropped blood cells: **RBC vs WBC**

The original dataset contains microscopic blood-smear images and bounding-box annotations. This notebook uses the annotations to crop individual cells and then trains a Convolutional Neural Network (CNN) to classify each crop.


## 1. Import Libraries

In [ ]:
import os
import zipfile
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)


## 2. Dataset Path

### Kaggle Notebook
If you add the dataset through **Add Input**, the notebook searches `/kaggle/input`.

### Local Jupyter Notebook
Set `DATASET_DIR` to the folder containing `images` and `annotations.csv`.


In [ ]:
# Try to automatically locate the dataset
possible_roots = [
    Path("/kaggle/input"),
    Path("."),
]

DATASET_DIR = None

for root in possible_roots:
    if root.exists():
        matches = list(root.rglob("annotations.csv"))
        if matches:
            DATASET_DIR = matches[0].parent
            break

# If automatic detection fails, manually set the path here:
# DATASET_DIR = Path(r"C:/Users/YourName/Downloads/blood-cell-detection-dataset")

if DATASET_DIR is None:
    raise FileNotFoundError(
        "Dataset not found. Add the Kaggle dataset as input or set DATASET_DIR manually."
    )

IMAGE_DIR = DATASET_DIR / "images"
ANNOTATION_FILE = DATASET_DIR / "annotations.csv"

print("Dataset directory:", DATASET_DIR)
print("Image directory:", IMAGE_DIR)
print("Annotation file:", ANNOTATION_FILE)


## 3. Load and Explore Annotations

In [ ]:
df = pd.read_csv(ANNOTATION_FILE)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())
print("\nMissing values:")
display(df.isnull().sum())


## 4. Standardize Annotation Columns

The original dataset is distributed with image names, bounding-box coordinates, and cell labels. This cell detects common column names so the notebook is easier to run in different environments.


In [ ]:
def find_column(columns, candidates):
    lower_map = {str(c).lower().strip(): c for c in columns}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    return None

image_col = find_column(df.columns, ["filename", "image", "image_name", "file_name"])
xmin_col  = find_column(df.columns, ["xmin", "x_min", "x1"])
ymin_col  = find_column(df.columns, ["ymin", "y_min", "y1"])
xmax_col  = find_column(df.columns, ["xmax", "x_max", "x2"])
ymax_col  = find_column(df.columns, ["ymax", "y_max", "y2"])
label_col = find_column(df.columns, ["label", "class", "class_name", "name"])

required = [image_col, xmin_col, ymin_col, xmax_col, ymax_col, label_col]

if any(col is None for col in required):
    raise ValueError(
        "Could not automatically identify annotation columns. "
        f"Available columns are: {df.columns.tolist()}"
    )

print("Image column:", image_col)
print("Bounding box:", xmin_col, ymin_col, xmax_col, ymax_col)
print("Label column:", label_col)

df = df[[image_col, xmin_col, ymin_col, xmax_col, ymax_col, label_col]].copy()
df.columns = ["image", "xmin", "ymin", "xmax", "ymax", "label"]

df["label"] = df["label"].astype(str).str.upper().str.strip()
print("\nClass distribution:")
display(df["label"].value_counts())


## 5. Visualize a Sample Image with Bounding Boxes

In [ ]:
from matplotlib.patches import Rectangle

sample_image_name = df["image"].iloc[0]
sample_rows = df[df["image"] == sample_image_name]

image_path = IMAGE_DIR / sample_image_name

# If image extension/path differs, try finding the file
if not image_path.exists():
    matches = list(IMAGE_DIR.rglob(Path(sample_image_name).name))
    if matches:
        image_path = matches[0]
    else:
        raise FileNotFoundError(f"Could not find image: {sample_image_name}")

img = Image.open(image_path).convert("RGB")

plt.figure(figsize=(7, 7))
plt.imshow(img)
ax = plt.gca()

for _, row in sample_rows.iterrows():
    width = row["xmax"] - row["xmin"]
    height = row["ymax"] - row["ymin"]
    rect = Rectangle(
        (row["xmin"], row["ymin"]),
        width,
        height,
        fill=False,
        linewidth=2
    )
    ax.add_patch(rect)
    ax.text(
        row["xmin"],
        max(0, row["ymin"] - 3),
        row["label"],
        fontsize=10,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.7)
    )

plt.title(f"Annotations: {sample_image_name}")
plt.axis("off")
plt.show()


## 6. Create Individual Cell Crops

Each annotated bounding box is converted into one image sample. Small padding is added around each cell so the CNN can learn some surrounding context.


In [ ]:
IMG_SIZE = 64
PADDING = 3

X = []
y = []

label_names = sorted(df["label"].unique())
label_to_id = {label: idx for idx, label in enumerate(label_names)}

print("Label mapping:", label_to_id)

grouped = df.groupby("image")

for image_name, rows in grouped:
    image_path = IMAGE_DIR / image_name

    if not image_path.exists():
        matches = list(IMAGE_DIR.rglob(Path(image_name).name))
        if matches:
            image_path = matches[0]
        else:
            print(f"Skipping missing image: {image_name}")
            continue

    image = Image.open(image_path).convert("RGB")
    width, height = image.size

    for _, row in rows.iterrows():
        xmin = max(0, int(row["xmin"]) - PADDING)
        ymin = max(0, int(row["ymin"]) - PADDING)
        xmax = min(width, int(row["xmax"]) + PADDING)
        ymax = min(height, int(row["ymax"]) + PADDING)

        if xmax <= xmin or ymax <= ymin:
            continue

        crop = image.crop((xmin, ymin, xmax, ymax))
        crop = crop.resize((IMG_SIZE, IMG_SIZE))

        X.append(np.array(crop))
        y.append(label_to_id[row["label"]])

X = np.array(X, dtype=np.float32) / 255.0
y = np.array(y, dtype=np.int32)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Classes:", label_names)


## 7. Visualize Cropped Cells

In [ ]:
plt.figure(figsize=(12, 8))

sample_indices = np.random.choice(len(X), min(12, len(X)), replace=False)

for i, idx in enumerate(sample_indices):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X[idx])
    plt.title(label_names[y[idx]])
    plt.axis("off")

plt.tight_layout()
plt.show()


## 8. Train, Validation and Test Split

The dataset is highly imbalanced because it contains many more RBC annotations than WBC annotations. We use stratified splitting and class weights during training.


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=SEED,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))


In [ ]:
# Calculate class weights to reduce the effect of class imbalance
class_counts = np.bincount(y_train)
total = len(y_train)
num_classes = len(class_counts)

class_weight = {
    i: total / (num_classes * count)
    for i, count in enumerate(class_counts)
}

print("Class counts:", class_counts)
print("Class weights:", class_weight)


## 9. Build the CNN Model

In [ ]:
num_classes = len(label_names)

model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Dropout(0.30),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.40),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


## 10. Train the Model

In [ ]:
early_stopping = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    class_weight=class_weight,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)


## 11. Training and Validation Performance

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CNN Loss")
plt.legend()
plt.grid(True)
plt.show()


## 12. Evaluate the Model

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

y_prob = model.predict(X_test)
y_pred = np.argmax(y_prob, axis=1)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_names,
        digits=4
    )
)


## 13. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_names,
    yticklabels=label_names
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()


## 14. Display Predictions

In [ ]:
plt.figure(figsize=(12, 10))

sample_indices = np.random.choice(len(X_test), min(12, len(X_test)), replace=False)

for i, idx in enumerate(sample_indices):
    predicted_label = label_names[y_pred[idx]]
    true_label = label_names[y_test[idx]]
    confidence = np.max(y_prob[idx]) * 100

    plt.subplot(3, 4, i + 1)
    plt.imshow(X_test[idx])
    plt.title(
        f"True: {true_label}\nPred: {predicted_label} ({confidence:.1f}%)",
        fontsize=9
    )
    plt.axis("off")

plt.tight_layout()
plt.show()


## 15. Save the Trained CNN Model

In [ ]:
MODEL_PATH = "blood_cell_cnn.keras"
model.save(MODEL_PATH)

print(f"Model saved successfully as: {MODEL_PATH}")


# Conclusion

A CNN model was developed to classify individual blood-cell crops generated from the bounding-box annotations in the Blood Cell Detection Dataset.

### Project Pipeline
1. Load blood smear images and annotations.
2. Extract individual cell images using bounding boxes.
3. Resize and normalize cell images.
4. Split the data into training, validation, and testing sets.
5. Handle class imbalance using class weights.
6. Train a CNN model.
7. Evaluate the model using accuracy, a classification report, and a confusion matrix.
8. Save the trained model.

> **Note:** This notebook performs **CNN-based cell classification** after cropping cells using the provided annotations. The original dataset itself is primarily an **object-detection dataset**, so direct full-image RBC/WBC classification would not correctly use its labels.
